# Numerical implementation notes and scratch work

We aim to describe and implement drafts of the components for a (non-realtime) MPC algorithm for the motion cueing Stewart platform problem.

In [ ]:
from __future__ import annotations

In [ ]:
%matplotlib ipympl

import time
import dataclasses
import functools
import typing as tp
import numpy as np
import sympy as sp
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import scipy.optimize as sci_opt
import scipy.sparse.linalg as sci_sp_lin
import IPython.display as ipd
import tqdm

import exp_mpc.stewart_min.const as const
import exp_mpc.stewart_min.utils as utils
import exp_mpc.stewart_min.quartic_cost as quartic_cost
import exp_mpc.stewart_min.spec as spec
import exp_mpc.stewart_min.viz as viz

jax.config.update("jax_enable_x64", True)

## jax discrete integration

Simply, we want to compute recursively
\begin{align*}
  v_{k + 1} &= v_k + \Delta t \, a_{k + 1} \\
  x_{k + 1} &= x_k + \Delta t \, v_{k + 1}.
\end{align*}
Explicitly, this is because, definitionally, we have
$$
  a_{k + 1} = \frac{v_{k + 1} - v_k}{\Delta t} \quad\text{and}\quad v_{k + 1} = \frac{x_{k + 1} - x_k}{\Delta t}.
$$

In [ ]:
@jax.jit
def _discrete_1d_euler(
    x0: float,
    v0: float,
    a: jax.Array
) -> tuple[jax.Array, jax.Array]:
    v = jnp.cumsum(jnp.concatenate([jnp.array([v0]), const.dt * a]))
    x = jnp.cumsum(jnp.concatenate([jnp.array([x0]), const.dt * v.at[1:].get()]))
    return x, v


In [ ]:
a = jnp.arange(1, 10, dtype=float)
state0 = 0.0
v0 = 0.0
# %timeit _discrete_1d_euler(x0, v0, a)
x, v = _discrete_1d_euler(state0, v0, a)
x, v
# fig, ax = plt.subplots(2, 1, figsize=(8, 6))
# ax[0].plot(x, label="position")
# ax[1].plot(v, label="velocity")
# for a in ax:
#     a.legend()
#     a.grid()
#     a.set_xlabel("time")
# plt.show()

## jax discrete integration for non-uniform times

Eventually, we want to integrate over a non-uniform time grid.
Specifically, given the usual discrete time update rule, we want a single computation that projects the output into the future, e.g., we want a formula for $x_{k + n} f(x_k, v_k, a_k)$ where $a_k$ is applied on the index (time) intervals $[k, k + 1], \ldots, [k + n - 1, k + n]$.
It is not difficult to check that the following linear time-invariant linear ODE conforms with the update rule over a single time step $\Delta t$:
$$
\begin{bmatrix} \dot{x} \\ \dot{v} \end{bmatrix} = \begin{bmatrix} 0 & 1 \\ 0 & 0 \end{bmatrix} \begin{bmatrix} x \\ v \end{bmatrix} + \begin{bmatrix} \Delta t / 2 \\ 1 \end{bmatrix} a.
$$
where we have a single discrete update of
$$
x_{k + 1} := x_k + \Delta t \, v_{k + 1} = x_k + \Delta t \, v_k + (\Delta t)^2 \, a_k
$$
(no factor of $\frac{1}{2}$).

Note that the differential equation conforms with the discrete update after one time step.
To show that this holds after multiple time steps, we may proceed inductively, time-step by time-step, in the case of constant acceleration.
So,
$$
  v(t + T) = v(t) + T \, a \quad\text{and}\quad x(t + T) = x(t) + T \, v(t) + \frac{1}{2} \, T^2 \, a + \frac{1}{2} \, T \, \Delta t \, a
$$
e.g., for $T = k \, \Delta t$,
$$
  v(t + k \, \Delta t) = v(t) + k \, \Delta t \, a
$$
and
\begin{align*}
  x(t + k \, \Delta t) &= x(t) + k \, \Delta t \, v(t) + \frac{1}{2} \, k^2 \, (\Delta t)^2 \, a + \frac{1}{2} \, k \, (\Delta t)^2 \, a \\
  &= x(t) + k \, \Delta t \, v(t) + \frac{1}{2} \, (k^2 + k) \, (\Delta t)^2 \, a.
\end{align*}

For simple NumPy code, we observe that
$$
  x(t + k \, \Delta t) = x(t) + k \, \Delta t \, v(t + k \, \Delta t) + \frac{1}{2} \, (k - k^2) \, (\Delta t)^2 \, a.
$$
(Note the minus sign.)

In [ ]:
@jax.jit
def _discrete_1d_nonuniform_euler(
    x0: float,
    v0: float,
    a: jax.Array,
    gaps: jax.Array,
) -> tuple[jax.Array, jax.Array]:
    # cf. https://stackoverflow.com/questions/37726830/how-to-determine-if-a-number-is-any-type-of-int-core-or-numpy-signed-or-not
    assert jnp.issubdtype(gaps.dtype, jnp.integer)
    assert jnp.issubdtype(a.dtype, jnp.floating)
    a = jnp.ravel(a)
    gaps = jnp.ravel(gaps)
    assert a.shape == gaps.shape
    v = jnp.cumsum(jnp.concatenate([jnp.array([v0]), const.dt * gaps * a]))
    x_vel = const.dt * gaps * v.at[1:].get()
    x_acc = 0.5 * (gaps - gaps**2) * (const.dt**2) * a
    x = jnp.cumsum(jnp.concatenate([jnp.array([x0]), x_vel + x_acc]))
    return x, v

In [ ]:
gaps = jnp.array([1, 2, 3, 4, 5, 6, 7, 8, 9])
state0 = 0.0
v0 = 0.0
a = jnp.array([1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=float)
# %timeit _discrete_1d_nonuniform_euler(x0, v0, a, gaps)
_discrete_1d_nonuniform_euler(state0, v0, a, gaps)

In [ ]:
gaps = jnp.ones(9, dtype=int)
state0 = 0.0
v0 = 0.0
a = jnp.array([1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=float)
x_non, v_non = _discrete_1d_nonuniform_euler(state0, v0, a, gaps)
x_uni, v_uni = _discrete_1d_euler(state0, v0, a)
assert jnp.allclose(x_non, x_uni)
assert jnp.allclose(v_non, v_uni)


## brute force check for non-uniform discrete integration

In [ ]:
def _discrete_1d_nonuniform_brute(
    x0: float,
    v0: float,
    a: jax.Array,
    gaps: jax.Array,
) -> tuple[jax.Array, jax.Array]:
    assert jnp.issubdtype(gaps.dtype, jnp.integer)
    assert jnp.issubdtype(a.dtype, jnp.floating)
    a = jnp.ravel(a)
    gaps = jnp.ravel(gaps)
    assert a.shape == gaps.shape
    
    # all computations will be performed in the default python datatypes, which
    #  are quite expressive, but slow
    al = [float(acc) for acc in a]
    gl = [int(g) for g in gaps]
    assert len(al) == len(gl)
    al = sum([[al[i]] * gl[i] for i in range(len(al))], start=[])
    assert len(al) == sum(gl)
    x = [x0]
    v = [v0]
    for acc in al:
        v.append(v[-1] + const.dt * acc)
        x.append(x[-1] + const.dt * v[-1])
    
    indices = [0]
    for i in range(len(gl)):
        indices.append(indices[-1] + gl[i])
    x = [x[i] for i in indices]
    v = [v[i] for i in indices]
    return jnp.array(x), jnp.array(v)


In [ ]:
gaps = jnp.array([1, 2, 3, 4, 5, 6, 7, 8, 9])
state0 = 0.0
v0 = 0.0
a = jnp.array([1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=float)
x_brute, v_brute = _discrete_1d_nonuniform_brute(state0, v0, a, gaps)
x_non, v_non = _discrete_1d_nonuniform_euler(state0, v0, a, gaps)
assert jnp.allclose(v_brute, v_non)
assert jnp.allclose(x_brute, x_non)

## MPC bookkeeping

We want to bookkeep the variables productively for the Stewart platform MPC problem.
The SciPy optimization routines expect a flat array.
We produce two classes: a state class and a control class.
Because we have such a simple double integrator model, we implement something akin to a single shooting method.
This drastically reduces the number of variables in our optimization routine.

In [ ]:
@jax.tree_util.register_dataclass
@dataclasses.dataclass
class State:
    x: jax.Array
    y: jax.Array
    z: jax.Array
    roll: jax.Array
    pitch: jax.Array
    yaw: jax.Array
    x_dot: jax.Array
    y_dot: jax.Array
    z_dot: jax.Array
    roll_dot: jax.Array
    pitch_dot: jax.Array
    yaw_dot: jax.Array

    def __iter__(self) -> tp.Iterator[jax.Array]:
        for field in dataclasses.fields(self):
            yield getattr(self, field.name)
    
    def __getitem__(self, key: int | slice) -> "State":
        return State(*[arr.at[key].get() for arr in self])

    def xyz(self) -> jax.Array:
        """Get the x, y, z coordinates of the state points."""
        # enforce 1d
        x = self.x.reshape(-1)
        y = self.y.reshape(-1)
        z = self.z.reshape(-1)
        return jnp.concatenate([x, y, z])

    def flatten(self) -> jax.Array:
        """Flatten the state into a single array."""
        return jnp.concatenate([arr.reshape(-1) for arr in self])



In [ ]:
@jax.tree_util.register_dataclass
@dataclasses.dataclass
class Control:
    """Parallel array of control inputs."""

    x: jax.Array
    y: jax.Array
    z: jax.Array
    roll: jax.Array
    pitch: jax.Array
    yaw: jax.Array

    @classmethod
    def from_flat(cls, flat: jax.Array) -> "Control":
        """Convert a flat array to a control dataclass.

        We assume that the flat array is of the form:
            [x0, y0, z0, roll0, pitch0, yaw0,
             x1, y1, z1, roll1, pitch1, yaw1,
             ...]
        where the first three elements are the x, y and z coordinates of the
        first control point, the next three are the roll, pitch and yaw
        angles of the first control point, and so on for all control points.
        """
        assert flat.size % 6 == 0
        flat = jnp.reshape(flat, (-1, 6))
        return cls(
            x=flat[:, 0],
            y=flat[:, 1],
            z=flat[:, 2],
            roll=flat[:, 3],
            pitch=flat[:, 4],
            yaw=flat[:, 5],
        )

    def flatten(self) -> jax.Array:
        """Convert a control dataclass to a flat array."""
        return jnp.ravel(
            jnp.column_stack(
                [
                    self.x,
                    self.y,
                    self.z,
                    self.roll,
                    self.pitch,
                    self.yaw,
                ]
            )
        )

    def get_state(
        self,
        state0: jax.Array,
        gaps: tp.Optional[jax.Array] = None,
    ) -> State:
        """Convert a control dataclass to a state dataclass."""
        state0 = jnp.ravel(state0)
        assert state0.size == 12
        x0, y0, z0, roll0, pitch0, yaw0 = state0[:6]
        x_dot0, y_dot0, z_dot0, roll_dot0, pitch_dot0, yaw_dot0 = state0[6:]
        if gaps is None:
            gaps = jnp.ones(self.x.size, dtype=int)
        assert gaps.size == self.x.size
        _euler = _discrete_1d_nonuniform_euler
        x, x_dot = _euler(x0, x_dot0, self.x, gaps)
        y, y_dot = _euler(y0, y_dot0, self.y, gaps)
        z, z_dot = _euler(z0, z_dot0, self.z, gaps)
        roll, roll_dot = _euler(roll0, roll_dot0, self.roll, gaps)
        pitch, pitch_dot = _euler(pitch0, pitch_dot0, self.pitch, gaps)
        yaw, yaw_dot = _euler(yaw0, yaw_dot0, self.yaw, gaps)
        return State(
            x=x,
            y=y,
            z=z,
            roll=roll,
            pitch=pitch,
            yaw=yaw,
            x_dot=x_dot,
            y_dot=y_dot,
            z_dot=z_dot,
            roll_dot=roll_dot,
            pitch_dot=pitch_dot,
            yaw_dot=yaw_dot,
        )

    def xyz(self) -> jax.Array:
        """Get the x, y, z coordinates of the control points."""
        # enforce 1d
        x = self.x.reshape(-1)
        y = self.y.reshape(-1)
        z = self.z.reshape(-1)
        return jnp.concatenate([x, y, z])

    def __iter__(self) -> tp.Iterator[jax.Array]:
        for field in dataclasses.fields(self):
            yield getattr(self, field.name)
    
    def __getitem__(self, key: int | slice) -> "Control":
        return Control(*[arr.at[key].get() for arr in self])


## example framework, for testing

The following are mainly for testing the cost functions, while training.

In [ ]:
# example control and state
control_ref = jnp.arange(0, 5, dtype=float)
control = Control(
    x=control_ref,
    y=control_ref,
    z=control_ref,
    roll=control_ref,
    pitch=control_ref,
    yaw=control_ref,
)
state0 = jnp.zeros(12, dtype=float)
state = control.get_state(state0)
control, state

In [ ]:
acc_ref = jnp.array([1.0, 0.0, 10.0])  # m/s^2
acc_ref = jnp.tile(A=acc_ref, reps=(control.x.size, 1))

omega_ref = jnp.array([0.0, -1.0, 0.0])  # rad/s
omega_ref = jnp.tile(A=omega_ref, reps=(control.x.size, 1))

acc_ref, omega_ref

In [ ]:
# acc_cost = quartic_cost.QuarticCost.from_bounds(
#     margins=[0.2, 0.1],
#     sizes=[1.0, 2**3, 2**8],
#     low=-const.max_cart_acc,
#     high=const.max_cart_acc,
# )
# omega_cost = quartic_cost.QuarticCost.from_bounds(
#     margins=[0.2, 0.1],
#     sizes=[1.0, 2**3, 2**8],
#     low=-const.max_angle_acc,
#     high=const.max_angle_acc,
# )

# technically, the zero (center) of leg_cost is not at the leg midpoint, but
#  rather the squared leg midpoint
# there does't seem to be that large of a difference between the two, and the
#  cost is supposed to be close to zero everywhere anyways
leg_cost = quartic_cost.QuarticCost.from_bounds(
    margins=[0.2, 0.1],
    sizes=[1.0, 2**3, 2**8],
    low=const.leg_min**2,  # square
    high=const.leg_max**2,  # square
)

joint_angle_cost = quartic_cost.QuarticCost.from_bounds(
    margins=[0.2, 0.1],
    sizes=[1.0, 2**3, 2**8],
    low=-const.joint_max_angle,
    high=const.joint_max_angle,
)

yaw_cost = quartic_cost.QuarticCost.from_bounds(
    margins=[0.2, 0.1],
    sizes=[1.0, 2**3, 2**8],
    low=-const.max_yaw,
    high=const.max_yaw,
)


In [ ]:
const.leg_mid**2

In [ ]:
const.leg_min**2

In [ ]:
const.leg_max**2

In [ ]:
(const.leg_min**2 + const.leg_max**2) / 2

## Cost Outline

We outline the components needed for our cost function.
1. Acceleration in the head (moving) frame
2. Angular velocity in the head (moving) frame
3. Boundary leg cost
4. Boundary joint angle cost
5. Yaw cost

These functions should be implemented in JAX, and they should include wrappers that input and output NumPy arrays (for the most compatibility with optimization libraries).
We need to be careful about scaling.

### Head Acceleration Cost

Let $x_{\mathrm{w}}$, $x_{\mathrm{r}}$, and $x_{\mathrm{h}}$ denote the world, robot (fixed), and head (moving) frame coordinates, respectively.
Then we have
$$
  x_{\mathrm{w}} = R \, x_{\mathrm{r}} + \Delta \quad\text{and}\quad x_{\mathrm{r}} = x_{\mathrm{h}} + \Delta_{\mathrm{h}}
$$
with $\Delta_{\mathrm{h}}$ constant.
Namely, we are controlling the table, so our head accelerations (in the world frame) include a centrifugal force:
$$
  \ddot{x}_{\mathrm{w}} = \ddot{R} \, (x_{\mathrm{h}} + \Delta_{\mathrm{h}}) + \ddot{\Delta}.
$$
We also have gravity $g = [0.0, 0.0, 9.81]$.
Translating into the head frame ($x_{\mathrm{h}} \mapsto x_{\mathrm{r}}$ instead of $x_{\mathrm{h}} \mapsto x_{\mathrm{w}}$), we have acceleration
$$
  R^{\operatorname{T}} \, (\ddot{R} \, \Delta_{\mathrm{h}} + \ddot{\Delta} + g).
$$
(I always thought that this extra translation is kind of subtle.
I will make a note about this somewhere, eventually.)

In [ ]:
def _get_R(state: State) -> jax.Array:
    """Get the rotation matrix from the state."""
    return utils._get_R(
        phi=state.roll,
        theta=state.pitch,
        psi=state.yaw,
    )


def _get_R_T(state: State) -> jax.Array:
    """Get the rotation matrix from the state."""
    return jnp.transpose(_get_R(state))


def _get_R_dot2(state: State, control: Control) -> jax.Array:
    """Get the rotation matrix 2nd derivative from the state."""
    return utils._get_R_dot2(
        phi=state.roll,
        theta=state.pitch,
        psi=state.yaw,
        phi_dot=state.roll_dot,
        theta_dot=state.pitch_dot,
        psi_dot=state.yaw_dot,
        phi_dot2=control.roll,
        theta_dot2=control.pitch,
        psi_dot2=control.yaw,
    )


# @jax.jit
def _head_xyz_acc_cost(
    acc_ref: jax.Array,
    state0: jax.Array,
    control: Control,
) -> jax.Array:
    assert acc_ref.ndim == 2
    assert acc_ref.shape[1] == 3
    assert acc_ref.shape[0] == control.x.size

    state = control.get_state(state0)

    def _single(_ref: jax.Array, _state: State, _control: Control) -> jax.Array:
        """Cost for a single input pairing."""
        _R_T = _get_R_T(_state)
        _R_dot2 = _get_R_dot2(_state, _control)
        _acc = _control.xyz()
        _world = _R_dot2 @ const.human_displacement + _acc + const.gravity
        _head = _R_T @ _world
        _diff = _head - _ref
        # print(f"_head={_head}")
        return _diff @ _diff  # sum squares

    # skip initial conditions in state
    # costs = jnp.array([_single(acc_ref[i], state[i + 1], control[i]) for i in range(control.x.size)])
    costs = jax.vmap(_single)(acc_ref, state[1:], control)
    return jnp.mean(costs)


In [ ]:
# _head_xyz_acc_cost(
#     acc_ref=acc_ref,
#     state0=state0,
#     control=Control.from_flat(last_control),
# )

### Head Angular Velocity Cost

The angular velocity in the head frame is given by $[\omega]_\times = R^{\mathrm{T}} \, \dot{R}$.
Because
$$
  [\omega]_\times = \begin{bmatrix} 0 & -\omega_z & \omega_y \\ \omega_z & 0 & -\omega_x \\ -\omega_y & \omega_x & 0 \end{bmatrix},
$$
we need the entries $(2, 1) \mapsto \omega_x$, $(0, 2) \mapsto \omega_y$, and $(1, 0) \mapsto \omega_z$.

In [ ]:
def _get_R_dot(state: State) -> jax.Array:
    """Rotation matrix derivative from the state."""
    return utils._get_R_dot(
        phi=state.roll,
        theta=state.pitch,
        psi=state.yaw,
        phi_dot=state.roll_dot,
        theta_dot=state.pitch_dot,
        psi_dot=state.yaw_dot,
    )


# @jax.jit
def _get_omega_cost(
    omega_ref: jax.Array,
    state0: jax.Array,
    control: Control,
) -> jax.Array:
    """Angular velocity cost."""
    assert omega_ref.ndim == 2
    assert omega_ref.shape[1] == 3
    assert omega_ref.shape[0] == control.x.size

    state = control.get_state(state0)

    def _single(_ref: jax.Array, _state: State) -> jax.Array:
        """Cost for a single input pairing."""
        _R_T = _get_R_T(_state)
        _R_dot = _get_R_dot(_state)
        _omega_mat = _R_T @ _R_dot
        _omega = jnp.array([
            _omega_mat[2, 1], _omega_mat[0, 2], _omega_mat[1, 0]
        ])
        _diff = _omega - _ref
        return _diff @ _diff

    # skip initial conditions in state
    costs = jax.vmap(_single)(omega_ref, state[1:])
    return jnp.mean(costs)


### Boundary Leg Cost

We use a custom quartic cost function to get something like asymptotic blow up near the boundary, without (hopefully) causing numerical instabilities for nonlinear numerical solver.

In [ ]:
def _get_squared_lengths(state: State) -> jax.Array:
    R = _get_R(state)
    t = state.xyz()

    lengths = []
    for i in range(6):
        top_i = R @ const.tops[i] + t
        diff = top_i - const.bots[i]
        lengths.append(diff @ diff)

    return jnp.array(lengths)


# @jax.jit
def _get_leg_boundary(
    cost: quartic_cost.QuarticCost,
    state0: jax.Array,
    control: Control,
) -> jax.Array:
    """Leg length cost."""
    state = control.get_state(state0)
    lengths = jnp.ravel(jax.vmap(_get_squared_lengths)(state[1:]))
    # costs = jnp.square(lengths - const.leg_mid**2)
    costs = jax.vmap(cost)(lengths)
    # print(lengths, const.leg_min**2, const.leg_max**2)
    # print(costs)
    # return jnp.mean(costs)
    return jnp.mean(costs)

In [ ]:
# Control.from_flat(last_control).get_state(state0).x

In [ ]:
# _get_leg_boundary(
#     cost=leg_cost,
#     state0=state0,
#     control=Control.from_flat(last_control),
# )

### Joint Angle Boundary Cost

In [ ]:
def _get_joint_angles(state: State) -> jax.Array:
    return jnp.concatenate(utils._angle_joint(
        x=state.x,
        y=state.y,
        z=state.z,
        phi=state.roll,
        theta=state.pitch,
        psi=state.yaw,
    ))


# @jax.jit
def _get_joint_angle_boundary(
    cost: quartic_cost.QuarticCost,
    state0: jax.Array,
    control: Control,
) -> jax.Array:
    """Joint angle cost.
    
    This is about 3 times more expensive to compute than the other
    cost functions (including boundary cost functions).
    """
    state = control.get_state(state0)
    angles = jnp.ravel(jax.vmap(_get_joint_angles)(state[1:]))
    return jnp.mean(jax.vmap(cost)(angles))

### Yaw Cost

In [ ]:
# @jax.jit
def _get_yaw_boundary(
    cost: quartic_cost.QuarticCost,
    state0: jax.Array,
    control: Control,
) -> jax.Array:
    """Yaw cost.
    
    Cheapest cost function to compute.
    """
    state = control.get_state(state0)
    return jnp.mean(jax.vmap(cost)(state.yaw[1:]))


In [ ]:
# @jax.jit
def _control_cost(
    control: Control,
) -> jax.Array:
    """Control cost."""
    return jnp.mean(jnp.square(control.flatten()))  # sum squares

### Total Cost, and Wrappers

In [ ]:
# @jax.jit
def _cost(
    acc_ref: jax.Array,
    omega_ref: jax.Array,
    leg_cost: quartic_cost.QuarticCost,
    joint_angle_cost: quartic_cost.QuarticCost,
    yaw_cost: quartic_cost.QuarticCost,
    state0: jax.Array,
    control: Control,
) -> jax.Array:
    """Cost function."""
    cost = jnp.array(0.0)
    cost = cost + _head_xyz_acc_cost(acc_ref, state0, control)
    cost = cost + _get_omega_cost(omega_ref, state0, control)
    cost = cost + _get_leg_boundary(leg_cost, state0, control)  # * 5e2
    cost = cost + _get_joint_angle_boundary(joint_angle_cost, state0, control)
    cost = cost + _get_yaw_boundary(yaw_cost, state0, control)
    cost = cost + _control_cost(control)
    return cost


@jax.jit
def _cost_flat(
    acc_ref: jax.Array,
    omega_ref: jax.Array,
    leg_cost: quartic_cost.QuarticCost,
    joint_angle_cost: quartic_cost.QuarticCost,
    yaw_cost: quartic_cost.QuarticCost,
    state0: jax.Array,
    control_flat: jax.Array,
) -> jax.Array:
    control = Control.from_flat(control_flat)
    return _cost(
        acc_ref,
        omega_ref,
        leg_cost,
        joint_angle_cost,
        yaw_cost,
        state0,
        control,
    )


@jax.jit
def _cost_jac(
    acc_ref: jax.Array,
    omega_ref: jax.Array,
    leg_cost: quartic_cost.QuarticCost,
    joint_angle_cost: quartic_cost.QuarticCost,
    yaw_cost: quartic_cost.QuarticCost,
    state0: jax.Array,
    control_flat: jax.Array
) -> jax.Array:
    cost = functools.partial(
        _cost_flat,
        acc_ref,
        omega_ref,
        leg_cost,
        joint_angle_cost,
        yaw_cost,
        state0,
    )
    return jax.grad(cost)(control_flat)
    

@jax.jit
def _cost_hessp(
    acc_ref: jax.Array,
    omega_ref: jax.Array,
    leg_cost: quartic_cost.QuarticCost,
    joint_angle_cost: quartic_cost.QuarticCost,
    yaw_cost: quartic_cost.QuarticCost,
    state0: jax.Array,
    control_flat: jax.Array,
    v: jax.Array,
) -> jax.Array:
    cost = functools.partial(
        _cost_flat,
        acc_ref,
        omega_ref,
        leg_cost,
        joint_angle_cost,
        yaw_cost,
        state0,
    )
    # primals and tangents should be _tuples_ of arrays
    primals = (jnp.array(control_flat),)
    tangents = (jnp.array(v),)
    res = jax.jvp(jax.grad(cost), primals, tangents)
    return res[1]


@jax.jit
def _cost_hess_mat(
    acc_ref: jax.Array,
    omega_ref: jax.Array,
    leg_cost: quartic_cost.QuarticCost,
    joint_angle_cost: quartic_cost.QuarticCost,
    yaw_cost: quartic_cost.QuarticCost,
    state0: jax.Array,
    control_flat: jax.Array,
) -> jax.Array:
    cost = functools.partial(
        _cost_flat,
        acc_ref,
        omega_ref,
        leg_cost,
        joint_angle_cost,
        yaw_cost,
        state0,
    )
    return jax.hessian(cost)(control)


def get_cost(
    acc_ref: jax.Array,
    omega_ref: jax.Array,
    leg_cost: quartic_cost.QuarticCost,
    joint_angle_cost: quartic_cost.QuarticCost,
    yaw_cost: quartic_cost.QuarticCost,
    state0: jax.Array,
    hess_type: str = "hessp",  # hessp, hess_lin_op, hess_mat
) -> tuple[tp.Callable, tp.Callable, tp.Callable]:
    """Get the cost function."""
    params = (acc_ref, omega_ref, leg_cost, joint_angle_cost, yaw_cost, state0)
    cost = functools.partial(_cost_flat, *params)
    cost_jac = functools.partial(_cost_jac, *params)
    cost_hessp = functools.partial(_cost_hessp, *params)
    cost_hess_mat = functools.partial(_cost_hess_mat, *params)

    def cost_wrapper(control_flat: np.ndarray) -> float:
        control = jnp.array(control_flat)
        return float(cost(control))

    def cost_jac_wrapper(control_flat: np.ndarray) -> np.ndarray:
        control = jnp.array(control_flat)
        return np.array(cost_jac(control))

    def cost_hessp_wrapper(
        control_flat: np.ndarray,
        v: np.ndarray,
    ) -> np.ndarray:
        # primals and tangents should be _tuples_ of arrays
        return np.array(cost_hessp(jnp.array(control_flat), jnp.array(v)))

    def cost_hess_lin_op_wrapper(
        control_flat: np.ndarray,
    ) -> sci_sp_lin.LinearOperator:
        matvec = functools.partial(cost_hessp_wrapper, control_flat)
        return sci_sp_lin.LinearOperator(
            shape=(control_flat.size, control_flat.size),
            matvec=matvec,  # type: ignore
            dtype=np.float64,
        )

    def cost_hess_mat_wrapper(control_flat: np.ndarray) -> np.ndarray:
        control = jnp.array(control_flat)
        return np.array(cost_hess_mat(control))

    if hess_type == "hessp":
        cost_hess_wrapper = cost_hessp_wrapper
    elif hess_type == "hess_lin_op":
        cost_hess_wrapper = cost_hess_lin_op_wrapper
    elif hess_type == "hess_mat":
        cost_hess_wrapper = cost_hess_mat_wrapper
    else:
        raise ValueError(f"Unknown hess_type: {hess_type}")

    return cost_wrapper, cost_jac_wrapper, cost_hess_wrapper


## Compare to ACADOS

Namely, we use a long horizon, without any gaps.

In [ ]:
T = 4.0  # s
# T = 1.25  # s
num_steps = int(T / const.dt)
n = 40
# n = 100

state0 = jnp.zeros(12, dtype=float)

acc_ref = jnp.array([1.0, 0.0, 0.0]) + const.gravity  # m/s^2
acc_ref = jnp.tile(A=acc_ref, reps=(n, 1))
omega_ref = jnp.array([0.0, 0.0, 0.0])  # rad/s
omega_ref = jnp.tile(A=omega_ref, reps=(n, 1))

margins = [0.2, 0.1]
sizes = [1.0, 2**3, 2**8]
# margins=[0.3, 0.1]
# sizes=[1.0, 2**3, 2**9]
leg_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=const.leg_min**2,  # square
    high=const.leg_max**2,  # square
)
joint_angle_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=-const.joint_max_angle,
    high=const.joint_max_angle,
)
yaw_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=-const.max_yaw,
    high=const.max_yaw,
)


In [ ]:
acados_get_cost = functools.partial(
    get_cost,
    acc_ref=acc_ref,
    omega_ref=omega_ref,
    leg_cost=leg_cost,
    joint_angle_cost=joint_angle_cost,
    yaw_cost=yaw_cost,
    # state0=state0,
    hess_type="hessp",
    # hess_type="hess_mat",
)

In [ ]:
def train_step(
    state0: jax.Array, last_control: jax.Array
) -> tuple[jax.Array, jax.Array, spec.TableSol, sci_opt.OptimizeResult, float]:
    """Return next state, computed control (flat), and table solution."""
    cost, cost_jac, cost_hess = acados_get_cost(state0=state0)
    t0 = time.time()
    res = sci_opt.minimize(
        fun=cost,
        x0=np.array(last_control),
        method="L-BFGS-B",
        jac=cost_jac,
        # hess=cost_hess,
        # hessp=cost_hess,
        # method="CG",
        # method="SLSQP",
        options={
            # "maxiter": 16,
            # "maxls": 8,
        },
    )
    t1 = time.time()
    control = Control.from_flat(jnp.array(res.x))
    state = control.get_state(state0)
    table_sol = spec.TableSol(
        x=np.array(jnp.column_stack(list(state))),
        u=np.array(jnp.column_stack(list(control))),
        stats=spec.TableStats(
            time=res.nit,
            status=res.status,
            cost=res.fun,
        )
    )
    t_tot = t1 - t0
    return state[1].flatten(), control.flatten(), table_sol, res, t_tot


In [ ]:
def split_table_sol(
    table_sol: spec.TableSol,
) -> list[spec.TableSol]:
    table_sols = []
    for i in range(1, table_sol.x.shape[0] - 1):
        table_sol_i = spec.TableSol(
            x=table_sol.x[i : i + 2],
            u=table_sol.u[i : i + 1],
            stats=table_sol.stats,
        )
        table_sols.append(table_sol_i)
    return table_sols

In [ ]:
state0 = jnp.zeros(12, dtype=float)
last_control = jnp.zeros(6 * n, dtype=float)
times = []
sol_list = []
res_list = []

for _ in tqdm.tqdm(range(num_steps)):
    state0, last_control, table_sol, res, t_tot = train_step(state0, last_control)
    sol_list.append(table_sol)
    res_list.append(res)
    times.append(t_tot)

In [ ]:
sol_list[10].u[:, 0]

In [ ]:
sol_list[11].u[:, 0]

In [ ]:
# print(min(times) * 1e3, max(times) * 1e3, np.mean(times) * 1e3, np.std(times) * 1e3)
# print(np.mean([res.nit for res in res_list]), np.std([res.nit for res in res_list]))
# print(np.mean([res.nfev for res in res_list]), np.std([res.nfev for res in res_list]))
# print(np.mean([res.njev for res in res_list]), np.std([res.njev for res in res_list]))
# print(Control.from_flat(last_control).x)
print(Control.from_flat(last_control).pitch)
# print(all([res.success for res in res_list]), res_list[-1])

In [ ]:
plt.close("all")

In [ ]:
anim_fig = viz.animate_trajectory(
    trajectory=sol_list,
    sim_rate=1.0,
    fps=30,
)

In [ ]:
# sol_list.extend(split_table_sol(sol_list[-1])[1:])
acados_human_fig = viz.plot_human_trajectory(
    trajectory=sol_list,
    references={
        "xyz-acceleration": np.tile(A=acc_ref[0], reps=(num_steps, 1)),
        "angular-velocity": np.tile(A=omega_ref[0], reps=(num_steps, 1)),
    }
)

In [ ]:
acados_table_fig = viz.plot_cartesian_table_trajectory(
    trajectory=sol_list,
)